# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/09/15/ai-in-productio

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [6]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer

In [7]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [8]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'external site',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [1]:
"""" The response_format Parameter: The line response_format={"type": "json_object"} forces the AI model to reply with a valid JSON string.
The Parsing Line: The line json.loads(result) takes that JSON string response and converts it into a native Python dictionary.
The Expected Key: The print statement links['links'] shows that the code expects the JSON object to contain a specific key named "links"."""

'" The response_format Parameter: The line response_format={"type": "json_object"} forces the AI model to reply with a valid JSON string.\nThe Parsing Line: The line json.loads(result) takes that JSON string response and converts it into a native Python dictionary.\nThe Expected Key: The print statement links[\'links\'] shows that the code expects the JSON object to contain a specific key named "links".'

In [9]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [10]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter/X profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook page',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [11]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 16 relevant links


{'links': [{'type': 'company homepage', 'url': 'https://huggingface.co'},
  {'type': 'brand/about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'pricing', 'url': 'https://huggingface.co/pricing'},
  {'type': 'enterprise', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'endpoints product', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'models', 'url': 'https://huggingface.co/models'},
  {'type': 'spaces', 'url': 'https://huggingface.co/spaces'},
  {'type': 'datasets', 'url': 'https://huggingface.co/datasets'},
  {'type': 'learn', 'url': 'https://huggingface.co/learn'},
  {'type': 'docs', 'url': 'https://huggingface.co/docs'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingfa

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [12]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [13]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
bytedance-research/Lance
Updated
2 days ago
•
1.68k
•
801
tencent/Hy-MT2-1.8B
Updated
3 days ago
•
5.55k
•
745
NemoStation/Marlin-2B
Updated
5 days ago
•
7.29k
•
331
tencent/Hy-MT2-30B-A3B
Updated
3 days ago
•
1.49k
•
321
sapientinc/HRM-Text-1B
Updated
4 days ago
•
90k
•
277
Browse 2M+ models
S

In [14]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [15]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [16]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 23 relevant links


"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nbytedance-research/Lance\nUpdated\n2 days ago\n•\n1.68k\n•\n801\ntencent/Hy-MT2-1.8B\nUpdated\n3 days ago\n•\n5.55k\n•\n745\nN

In [17]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [18]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links


# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is a leading AI company and the vibrant open-source community building the future of machine learning. Their platform is a collaborative hub where ML researchers, developers, and enterprises come together to create, share, and innovate on models, datasets, and applications. Hugging Face empowers AI development by democratizing access to state-of-the-art machine learning tools and resources, fostering a spirit of openness and innovation.

---

## Core Offerings

- **Model Hub:** Browse and collaborate on over 2 million machine learning models across a variety of domains and languages.
- **Datasets:** Access over 500,000 curated datasets for training, testing, and validating AI models.
- **Spaces:** Host and explore ML applications and demos created by the community with ease.
- **Buckets & Storage:** Integrated storage solutions for efficient data management and inference tasks.
- **HuggingChat:** An AI chat interface designed to showcase conversational AI capabilities.
- **Enterprise Solutions:** Tailored support including Hugging Face PRO, enterprise-grade inference endpoints, and managed services.

---

## Community & Culture

Hugging Face thrives on its inclusive, mission-driven community that emphasizes:

- **Collaboration:** Facilitates open collaboration with unlimited sharing of public models, datasets, and apps.
- **Innovation:** Continuously updated contributions from thousands of AI researchers and engineers worldwide.
- **Democratization:** Making machine learning accessible and usable by everyone through open-source projects and educational content.
- **Transparency:** Regular publications, blog posts, and scientific papers that advance the global AI ecosystem.
- **Support:** Active forums, Discord channels, and dedicated community managers foster a supportive environment for learning and growth.

The team embodies a mission to “democratize good machine learning, one commit at a time,” highlighting their focus on quality, openness, and shared progress.

---

## Customers & Partners

Hugging Face serves a diverse and growing customer base:

- Researchers and academics utilizing the model and dataset hubs for cutting-edge AI research.
- AI practitioners and developers building next-generation applications.
- Enterprises deploying scalable AI solutions with professional support and custom inference endpoints.
- Organizations and teams leveraging collaborative tools to accelerate AI innovation internally.

---

## Careers at Hugging Face

Join a rapidly growing global team of 180+ dedicated professionals who are passionate about open-source AI and transforming the future of machine learning.

- Roles include engineering, research, community management, developer support, and enterprise sales.
- Emphasis on a collaborative, inclusive culture that values curiosity, openness, and continuous learning.
- Opportunity to work at the forefront of AI innovation, contributing to world-class open-source projects and enterprise AI solutions.
- Remote-friendly work environment with a strong, connected international community.

Interested in making an impact? Hugging Face invites talented individuals who share their mission to join them on their journey.

---

## Connect & Learn More

- Website: https://huggingface.co
- GitHub: https://github.com/huggingface
- Community: Join their active Discord and forums for real-time conversations.
- Blog & Papers: Stay updated on AI research and ecosystem developments.
- Contact: For press or business inquiries, reach out via their official contact channels.

---

**Hugging Face** — _The AI community building the future._

*Empowering the worldwide machine learning community through open collaboration, innovation, and inclusion.*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [19]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [20]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is a pioneering AI company dedicated to building the future through community collaboration. As the premier platform for machine learning, Hugging Face empowers a global community of developers, researchers, and enterprises to create, discover, and collaborate on cutting-edge AI models, datasets, and applications. 

With over **2 million models** and **500,000+ datasets**, Hugging Face provides an open, inclusive ecosystem that fuels innovation in artificial intelligence.

---

## What We Offer

### The Collaboration Platform  
- Host and collaborate on **unlimited public models, datasets, and applications**.  
- Access a vast and growing collection of AI models—such as natural language processing, computer vision, and more—from leading AI research and corporate labs.  
- Explore **Spaces**: share and run AI apps created by the community seamlessly.  
- Enable your teams with **Enterprise Solutions**, including dedicated support, inference endpoints, and scalable storage buckets.  

### Key Features  
- **Models:** Browse and contribute to over 2 million AI models updated frequently by a vibrant community.  
- **Datasets:** Access 500k+ public datasets to train and improve your AI systems.  
- **Spaces:** Discover and deploy interactive AI applications built by community members.  
- **HuggingChat:** Engage with AI-powered chat capabilities integrated into the platform.  
- **Enterprise Support:** Tailored services for businesses requiring robust AI deployment and scaling.  

---

## Our Community & Customers

Hugging Face thrives on its dynamic, global AI community including developers, researchers, data scientists, and enterprises from tech giants and startups alike. The platform fosters collaborative innovation, enabling users to share models and datasets freely to advance AI research and applications collectively.

Our customers range from individual AI enthusiasts to large organizations seeking reliable AI tools and infrastructure for production-scale deployment.

---

## Company Culture

At Hugging Face, collaboration and openness are at our core. We believe in building transparent, accessible AI tools that empower everyone. Our culture promotes:  
- **Community-driven development**  
- **Open science and sharing**  
- **Innovation through collaboration**  
- **Supportive and inclusive environment**  

---

## Careers at Hugging Face

Join us to be part of an innovative company shaping the future of AI. We look for passionate individuals who thrive in a fast-paced, community-oriented environment. Whether your expertise lies in machine learning research, software engineering, or product development, Hugging Face offers exciting roles to push the boundaries of what's possible with AI.

- Opportunities to work with a diverse, global team.  
- Contribute directly to widely used open-source projects.  
- A culture that values learning, creativity, and impact.  

Check our website regularly for new job openings and apply to help us democratize AI!

---

## Get Involved

- Explore thousands of models and datasets on our [Hugging Face Hub](https://huggingface.co).  
- Join our vibrant community on [Discord](https://discord.gg/huggingface) and [Forum](https://discuss.huggingface.co).  
- Collaborate via GitHub and contribute to open-source AI development.  

---

**Hugging Face**  
The AI community building the future, together.  

_Visit us at https://huggingface.co to learn more._  

---

**Brand Colors:**  
- Yellow: #FFD21E  
- Orange: #FF9D00  
- Gray: #6B7280  

---

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>